# Apprentissage par Renforcement avec Policy-Gradient : l'Agent CartPole

## Contexte et Objectifs

Ce notebook presente une implementation detaillee de l'algorithme REINFORCE, une methode fondamentale de la famille des Policy-Gradient. Contrairement aux methodes basees sur la valeur (comme le Q-Learning) qui apprennent une fonction de valeur, les methodes de Policy-Gradient apprennent directement une politique parametrisee, generalement un reseau de neurones.

Nous appliquons cet algorithme a l'environnement classique `CartPole-v1` de `gymnasium`, ou l'objectif est d'apprendre a un agent a equilibrer un poteau sur un chariot.

### Structure du Notebook :

1.  **Environnement `CartPole-v1` :** Presentation de l'environnement, de ses etats, de ses actions et de son objectif.
2.  **Le Reseau de Politique (Policy Network) :** Nous utilisons PyTorch pour definir un reseau de neurones simple qui prend en entree l'etat de l'environnement et produit une distribution de probabilite sur les actions possibles.
3.  **Algorithme REINFORCE :** Implementation de l'algorithme, qui consiste a augmenter la probabilite des actions ayant mene a de fortes recompenses.
4.  **Boucle d'Entrainement :** L'agent interagit avec l'environnement, collecte des trajectoires (sequences d'etats, d'actions et de recompenses) et met a jour son reseau de politique.
5.  **Visualisation de l'Apprentissage :** Un graphique montre l'evolution des recompenses obtenues par l'agent au fil du temps, illustrant sa progression.

_Derniere mise a jour : 2026-02-16_

In [ ]:
# --- 1. Installation des Dependances ---
%pip install -q gymnasium torch numpy matplotlib
print("Dependances installees.")

In [ ]:
# --- 2. Imports ---
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt
import logging

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## 3. Le Reseau de Politique (Policy Network)

Le cœur de notre agent est un simple reseau de neurones a deux couches. Il prend en entree une observation de l'environnement (4 valeurs continues) et sort des logits pour les deux actions possibles (aller a gauche ou a droite). Une fonction softmax est ensuite appliquee pour obtenir une distribution de probabilite.

In [ ]:
class PolicyNetwork(nn.Module):
    """Reseau de neurones pour representer la politique de l'agent."""
    def __init__(self, input_size, output_size):
        super(PolicyNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, output_size)
        )

    def forward(self, x):
        logits = self.net(x)
        # On retourne une distribution de probabilite sur les actions
        return Categorical(logits=logits)

## 4. L'Agent REINFORCE

Cet agent implemente l'algorithme REINFORCE. A chaque etape, il :
1.  Utilise son reseau de politique pour choisir une action.
2.  Stocke le logarithme de la probabilite de cette action et la recompense obtenue.
3.  A la fin d'un episode, calcule les recompenses actualisees (returns).
4.  Met a jour les poids de son reseau pour que les actions ayant mene a de bons returns deviennent plus probables.

In [ ]:
class ReinforceAgent:
    def __init__(self, env, learning_rate=0.01, gamma=0.99):
        self.env = env
        self.gamma = gamma
        state_dim = env.observation_space.shape[0]
        action_dim = env.action_space.n
        
        self.policy_network = PolicyNetwork(state_dim, action_dim)
        self.optimizer = optim.Adam(self.policy_network.parameters(), lr=learning_rate)
        
        self.log_probs = []
        self.rewards = []

    def select_action(self, state):
        state_tensor = torch.from_numpy(state).float().unsqueeze(0)
        action_dist = self.policy_network(state_tensor)
        action = action_dist.sample()
        
        # Stocker le log de la probabilite de l'action choisie
        self.log_probs.append(action_dist.log_prob(action))
        return action.item()

    def update_policy(self):
        # Calculer les recompenses actualisees (returns)
        returns = []
        G = 0
        for r in reversed(self.rewards):
            G = r + self.gamma * G
            returns.insert(0, G)
        
        returns = torch.tensor(returns)
        # Normaliser les returns pour stabiliser l'entrainement
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)
        
        # Calculer la perte
        policy_loss = []
        for log_prob, G in zip(self.log_probs, returns):
            policy_loss.append(-log_prob * G) # Ponderer par le return
        
        # Mettre a jour le reseau
        self.optimizer.zero_grad()
        policy_loss = torch.cat(policy_loss).sum()
        policy_loss.backward()
        self.optimizer.step()
        
        # Vider les memoires pour le prochain episode
        self.log_probs = []
        self.rewards = []

## 5. Boucle d'Entrainement et Visualisation

Nous entrainons l'agent sur un certain nombre d'episodes. Le but est d'atteindre une recompense moyenne elevee, ce qui signifie que l'agent parvient a maintenir le poteau en equilibre de plus en plus longtemps.

In [ ]:
# --- Parametres d'entrainement ---
env = gym.make('CartPole-v1')
agent = ReinforceAgent(env)
num_episodes = 500
rewards_history = []

logger.info(f"Debut de l'entrainement pour {num_episodes} episodes...")

for episode in range(num_episodes):
    state, _ = env.reset()
    episode_reward = 0
    done = False
    
    while not done:
        action = agent.select_action(state)
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        agent.rewards.append(reward)
        episode_reward += reward
        
    agent.update_policy()
    rewards_history.append(episode_reward)
    
    if (episode + 1) % 50 == 0:
        avg_reward = np.mean(rewards_history[-50:])
        logger.info(f"Episode {episode + 1}/{num_episodes} | Recompense moyenne (50 derniers): {avg_reward:.2f}")

env.close()
logger.info("Entrainement termine.")

# --- Visualisation ---
plt.figure(figsize=(12, 6))
plt.plot(rewards_history, label='Recompense par Episode')
plt.plot(pd.Series(rewards_history).rolling(50).mean(), label='Moyenne Mobile (50 ep.)', color='red')
plt.title("Progression de la Recompense de l'Agent CartPole")
plt.xlabel("Episode")
plt.ylabel("Recompense Totale")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Marqueur d'execution pour garantir au moins une sortie
print('Notebook execute avec succes — ' + time.strftime('%Y-%m-%d %H:%M:%S'))